# Numerical resolution: particle in a box, particle on a ring

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print('Ready')

---
# Part I Discretisation of the Hamiltonian operator

## 1.1 Principle
We replace the second derivative with its finite difference approximation

$$\frac{d^2\psi}{dx^2}\bigg|_{x_i} \approx \frac{\psi_{i+1} - 2\psi_i + \psi_{i-1}}{\Delta x^2}$$

The Hamiltonian $\hat{H} = -\frac{\hbar^2}{2m}\frac{d^2}{dx^2}$ becomes the matrix (**in atomic units $\hbar = m = 1$**) :

$$2H_{ij} = \frac{1}{\Delta x^2}\begin{cases} +2 & \text{if } i=j \\ -1 & \text{if } |i-j|=1 \\ 0 & \text{else} \end{cases}$$

In [ ]:
#To visualize the structure of the hamiltonian matrices
# we take few points to keep the matrices small and readable
N_vis = 8

def box_matrix(N, L=1.0):
    #your code here
    pass


def ring_matrix(N):
    #your code here
    pass

H_box_vis  = box_matrix(N_vis)
H_ring_vis = ring_matrix(N_vis)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, H, titre in zip(axes,
        [H_box_vis, H_ring_vis],
        ['Box',
         'Ring']):
    mask = np.abs(H) > 1e-10
    im = ax.imshow(mask.astype(float), cmap='RdBu_r', vmin=-0.5, vmax=1.5)
    ax.set_xticks(range(N_vis))
    ax.set_yticks(range(N_vis))
    ax.set_xticklabels([str(i) for i in range(N_vis)])
    ax.set_yticklabels([str(i) for i in range(N_vis)])
    ax.set_title(titre, fontsize=10)

plt.suptitle(f'Structure of the hamiltonian matrices (N={N_vis} points)')
plt.tight_layout()
plt.show()

---
# Part II - Particule in a box

## 2.1 Diagonalization

In [ ]:
N_box = 500
L     = 1.0
x_box = np.linspace(0, L, N_box + 2)[1:-1]   # points

H_box  = box_matrix(N_box, L)
vals_box, vecs_box = np.linalg.eigh(H_box)
# Facteur 1/2 : H = (1/dx²)(...) corresponds to (-d²/dx²), energy = eigenvalue/2
E_num_box = vals_box / 2

## 2.2 Wave functions and densities

In [ ]:
colors = plt.cm.viridis(np.linspace(0.1, 0.9, 4))
fig, ax = plt.subplots(figsize=(6, 13))

# Wave functions
for i, col in enumerate(colors):
    psi_n   = vecs_box[:, i]
    # Normalisation and sign adjustment (arbitrary sign of eigenvectors)
    psi_n  /= np.sqrt(np.trapz(psi_n**2, x_box))
    if psi_n[N_box//4] < 0: psi_n = -psi_n
    offset  = E_num_box[i]/8
    ax.plot(x_box, psi_n + offset, color=col, linewidth=2, label=f'n={i+1}')
    #plot density
    rho_n = psi_n**2
    ax.fill_between(x_box, offset, rho_n + offset, alpha=0.2, color=col)
    ax.axhline(offset, color=col, linewidth=0.7, linestyle='--', alpha=0.5)
ax.set_xlabel('x / L')
ax.set_ylabel('ψ_n(x) + E_n  (offset)')
ax.set_title('Wave functions ψ_n(x) with energy offsets E_n')
ax.legend(fontsize=8)

plt.show()

---
# Parti III - Particule on a ring

## 3.1 Diagonalisation

In [ ]:
N_ring = 500
phi_ring = np.linspace(0, 2*np.pi, N_ring, endpoint=False)

H_ring  = ring_matrix(N_ring)
vals_ring, vecs_ring = np.linalg.eigh(H_ring)
E_num_ring = vals_ring / 2

## 3.2 Wave functions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

pairs = [(0, None, 'n=0  (non degenerate)'),
         (1, 2,   'n=±1 (degenerate)'),
         (3, 4,   'n=±2 (degenerate)')]

colors_ring = ['steelblue', 'tomato', 'seagreen']

for col_idx, (i1, i2, titre) in enumerate(pairs):
    col = colors_ring[col_idx]

    psi1 = vecs_ring[:, i1].copy()
    psi1 /= np.sqrt(np.trapz(psi1**2, phi_ring))

    # Fonctions d'onde
    ax = axes[0, col_idx]
    ax.plot(phi_ring, psi1, color=col, linewidth=2, label=f'vecteur {i1}')
    if i2 is not None:
        psi2 = vecs_ring[:, i2].copy()
        psi2 /= np.sqrt(np.trapz(psi2**2, phi_ring))
        ax.plot(phi_ring, psi2, color=col, linewidth=2,
                linestyle='--', label=f'vecteur {i2}')
    ax.set_title(titre, fontsize=9)
    ax.set_xlabel('φ')
    ax.set_ylabel('ψ(φ)')
    ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
    ax.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    # set yrange [-1:1]
    ax.set_ylim(-1, 1)

    # Density
    ax2 = axes[1, col_idx]
    rho1 = psi1**2
    ax2.plot(phi_ring, rho1, color=col, linewidth=2)
    ax2.fill_between(phi_ring, 0, rho1, alpha=0.2, color=col)
    if i2 is not None:
        rho2 = psi2**2
        ax2.plot(phi_ring, rho2, color=col, linewidth=2, linestyle='--')
        ax2.fill_between(phi_ring, 0, rho2, alpha=0.1, color=col)
        rho_sum = rho1 + rho2
        ax2.plot(phi_ring, rho_sum/2, 'k-', linewidth=1.5,
                 label='moyenne')
    ax2.set_xlabel('φ')
    ax2.set_ylabel('|ψ(φ)|²')
    ax2.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
    ax2.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
    ax2.grid(True, alpha=0.3)

plt.suptitle('Wave functions of the ring',
             fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# We play on the value of the connecting terms between the first and last points in the ring matrix to see how it affects the eigenvalues and eigenvectors.

def ring_matrix(N, coef=1.0):
    dphi = 2 * np.pi / N
    diag = 2 * np.ones(N) / dphi**2
    off  = -np.ones(N-1) / dphi**2
    H    = np.diag(diag) + np.diag(off, 1) + np.diag(off, -1)
    # Periodic conditions: connect the first and last points
    H[0, N-1] = -coef / dphi**2
    H[N-1, 0] = -coef / dphi**2
    return H

#Plot the solutions for coef in 0.0001, 0.1, 0.5, 1.0
coefs = [0.0001, 0.1, 0.5, 0.8, 0.9, 1.0]
fig, axes = plt.subplots(2, 3, figsize=(12, 10))
for ax, coef in zip(axes.flatten(), coefs):
    print(f'coef = {coef}')
    H = ring_matrix(N_ring, coef=coef)
    vals, vecs = np.linalg.eigh(H)
    E_num = vals / 2

    psi1 = vecs[:, 0].copy()
    psi1 /= np.sqrt(np.trapz(psi1**2, phi_ring))

    ax.plot(phi_ring, psi1, color='steelblue', linewidth=2)
    ax.set_title(f'connection coefficient = {coef}', fontsize=9)
    ax.set_xlabel('φ')
    ax.set_ylabel('ψ(φ)')
    ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
    ax.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1, 1)

    #plots the first eigenvalues as vertical lines
    for i in range(4):
        ax.axhline(E_num[i], color='tomato', linewidth=2, linestyle='--', alpha=0.5)
plt.suptitle('Effect of the connection coefficient on the ring solutions',
             fontsize=10)
plt.tight_layout()
plt.show()
